# Test Qwen trực tiếp với 3 keyframe

Upload 3 ảnh, gọi Qwen và in raw response. Không dùng retrieval hay JSON parser strict.

In [ ]:
from pathlib import Path

GIT_URL = "https://github.com/24122013/AIChallenge26_Multimodal_Agentic_Video_Retrieval_System.git"
GIT_BRANCH = "main"
QA_QUERY = "Chiếc xe đẩy em bé cạnh người phụ nữ có màu gì?"  # sửa câu hỏi
QA_MODEL = "Qwen/Qwen3.5-9B"
QA_MODEL_REVISION = "c202236235762e1c871ad0ccb60c8ee5ba337b9a"
QA_QUANTIZATION = "4bit"
DEVICE = "cuda"

WORKSPACE = Path("/content/ai_challenge26")
UPLOAD_ROOT = Path("/content/manual_keyframes")
MODEL_CACHE_ROOT = Path("/content/model_cache")
OUTPUT_ROOT = Path("/content/drive/MyDrive/AIChallenge26/qa_manual_3_images")


## 1. Chuẩn bị GPU, repo và model cache

In [ ]:
from google.colab import drive, userdata
import os, shutil, subprocess, sys, torch

drive.mount("/content/drive")
if not torch.cuda.is_available():
    raise RuntimeError("Chưa bật GPU: Runtime > Change runtime type > T4 GPU")
print({"gpu": torch.cuda.get_device_name(0), "memory_mib": torch.cuda.get_device_properties(0).total_memory // 1024**2})
github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Thiếu Colab secret GITHUB_TOKEN hoặc Notebook access chưa bật.")
hf_token = userdata.get("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
askpass = Path("/content/git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo x-access-token ;;\n  *) echo "$GITHUB_TOKEN" ;;\nesac\n', encoding="utf-8")
askpass.chmod(0o700)
if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
try:
    clone_env = {**os.environ, "GITHUB_TOKEN": github_token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"}
    subprocess.run(["git", "-c", "http.version=HTTP/1.1", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(WORKSPACE)], env=clone_env, check=True)
finally:
    askpass.unlink(missing_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
print("ready:", Path.cwd())


## 2. Upload đúng 3 ảnh keyframe

In [ ]:
from google.colab import files
from IPython.display import display
from PIL import Image

if UPLOAD_ROOT.exists():
    shutil.rmtree(UPLOAD_ROOT)
UPLOAD_ROOT.mkdir(parents=True)
uploaded = files.upload()
if len(uploaded) != 3:
    raise ValueError(f"Cần đúng 3 ảnh, nhận được {len(uploaded)} file")
evidence = []
for rank, (name, content) in enumerate(uploaded.items(), start=1):
    path = UPLOAD_ROOT / Path(name).name
    path.write_bytes(content)
    with Image.open(path) as image:
        image.verify()
    display(Image.open(path).convert("RGB"))
    evidence.append({"evidence_id": f"MANUAL_{rank}", "video_id": "manual_upload", "frame_id": path.name, "timestamp": float(rank), "image_path": str(path), "caption": "", "ocr_text": "", "objects": []})
print("Uploaded:", [item["frame_id"] for item in evidence])


## 3. Gọi Qwen và in đáp án thô

Lần đầu tải/lắp model có thể mất nhiều phút. Không interrupt cell này.

In [ ]:
import json
from backend.app.services.retrieval.qa_answerer import build_local_qwen_runner

runner = build_local_qwen_runner(model_name=QA_MODEL, model_revision=QA_MODEL_REVISION, device=DEVICE, quantization=QA_QUANTIZATION, cache_dir=MODEL_CACHE_ROOT / "qa_answer", max_new_tokens=256)
raw_response = runner(QA_QUERY, evidence, "unknown")
result = {"query": QA_QUERY, "qwen_called": True, "raw_response": raw_response, "evidence": evidence}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
result_path = OUTPUT_ROOT / "qa_manual_3_images_raw_response.json"
result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print("QWEN RAW RESPONSE:\n")
print(raw_response)
print("\nsaved:", result_path)
